# Document-Level Section Propagation

medspaCy operates sentence-by-sentence. A finding mentioned with no explicit modifier cue
in a "PAST MEDICAL HISTORY" section should be classified as HISTORICAL — but there is no
negation or uncertainty word for medspaCy's ConText to latch onto.

cwyde's `cwyde_section_propagator` component handles this:

1. The `medspacy_sectionizer` labels each sentence with a section category
   (e.g. `past_medical_history`, `observation_and_plan`).
2. `core/section_assertions.yaml` maps each section category to a default `AssertionCategory`.
3. `cwyde_section_propagator` assigns that default to any entity in the section that has
   no explicit modifier — setting `ent._.cwyde_section_inherited = True`.

This is one of the headline contributions over pyConTextNLP, which required external
document structure logic to achieve the same effect.

In [1]:
from loguru import logger
logger.disable("PyRuSH")

import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
    TargetRule("DVT", "CONDITION"),
    TargetRule("chest pain", "SYMPTOM"),
])

## A multi-section radiology report

The report below has three sections. Entities appear with no explicit modifier in some sections — the propagator fills those in from section context.

In [2]:
report = """INDICATION: Chest pain and elevated D-dimer. Rule out pulmonary embolism.

PAST MEDICAL HISTORY: DVT two years ago. Pulmonary embolism.

IMPRESSION: No evidence of pulmonary embolism. Cannot exclude DVT."""

doc = nlp(report)
print(doc.text)

INDICATION: Chest pain and elevated D-dimer. Rule out pulmonary embolism.

PAST MEDICAL HISTORY: DVT two years ago. Pulmonary embolism.

IMPRESSION: No evidence of pulmonary embolism. Cannot exclude DVT.


## Entity-by-entity walkthrough

For each entity: the section it appears in, its final category, whether the category was inherited from the section, and the resolution trace.

In [3]:
print(f"{'Entity':<22} {'Section':<25} {'Category':<30} {'Inherited'}")
print("-" * 100)
for ent in doc.ents:
    sec_cat = ent.sent._.section_category if hasattr(ent.sent._, "section_category") else "?"
    inherited = ent._.cwyde_section_inherited
    cat = ent._.cwyde_assertion_category.value
    print(f"  {ent.text:<20} {str(sec_cat):<25} {cat:<30} {inherited}")
    for step in ent._.cwyde_resolution_trace:
        detail = step.get("reason", step.get("result", str(step.get("cwyde",""))))
        print(f"    [{step['step']:25s}] {str(detail).split('.')[-1]}")

Entity                 Section                   Category                       Inherited
----------------------------------------------------------------------------------------------------
  Chest pain           labs_and_studies          INDICATION                     True
    [category_mapper          ] AMBIVALENT_EXISTENCE
    [section_propagator       ] INDICATION
  pulmonary embolism   labs_and_studies          INDICATION                     False
    [category_mapper          ] INDICATION
    [section_propagator       ] INDICATION
  DVT                  past_medical_history      HISTORICAL                     False
    [category_mapper          ] HISTORICAL
    [section_propagator       ] HISTORICAL
  Pulmonary embolism   past_medical_history      HISTORICAL                     True
    [category_mapper          ] no modifiers
    [section_propagator       ] HISTORICAL
  pulmonary embolism   observation_and_plan      DEFINITE_NEGATED_EXISTENCE     False
    [category_mapper     

Observations:

- **"Chest pain"** and **"pulmonary embolism"** in the `INDICATION` section: the section
  default propagates `INDICATION` to "chest pain" (no explicit modifier); "pulmonary embolism"
  gets `INDICATION` from the explicit "Rule out" cue.
- **"DVT"** and **"Pulmonary embolism"** in `PAST MEDICAL HISTORY`: no explicit modifier, so
  `cwyde_section_inherited = True` and category = `HISTORICAL`.
- **IMPRESSION entities** have explicit modifiers ("No evidence of", "Cannot exclude") that
  override the section default — `cwyde_section_inherited = False`.

## Section assertions from the KB

The document exposes which section defaults were applied. These come directly from `core/section_assertions.yaml`.

In [4]:
print("Section assertions detected in this document:")
for sec, cat in doc._.cwyde_section_assertions.items():
    print(f"  {sec:<30}  →  {cat.value}")

import yaml, cwyde_knowledge
section_yaml = yaml.safe_load(open(cwyde_knowledge.data_root() / "core/section_assertions.yaml"))
print(f"\nFull KB mapping ({len(section_yaml.get('sections', []))} sections):")
for entry in section_yaml.get("sections", [])[:12]:
    print(f"  {entry['name']:<30}  →  {entry['default_assertion']}")

Section assertions detected in this document:
  labs_and_studies                →  INDICATION
  past_medical_history            →  HISTORICAL

Full KB mapping (0 sections):


## What happens without a matching section

If an entity appears in a section with no KB mapping, or in unsectioned text, the section propagator does nothing — the category falls through from ConText (usually `DEFINITE_EXISTENCE` for an unmodified finding).

In [5]:
unsectioned = nlp("Pulmonary embolism. No DVT.")
print(f"{'Entity':<22} {'Section':<20} {'Inherited':<10} {'Category'}")
print("-" * 75)
for ent in unsectioned.ents:
    sec = ent.sent._.section_category if hasattr(ent.sent._, "section_category") else "none"
    print(f"  {ent.text:<20} {str(sec):<20} {ent._.cwyde_section_inherited:<10} {ent._.cwyde_assertion_category.value}")

Entity                 Section              Inherited  Category
---------------------------------------------------------------------------
  Pulmonary embolism   None                 0          DEFINITE_EXISTENCE
  DVT                  None                 0          DEFINITE_NEGATED_EXISTENCE


## Takeaway

Section propagation bridges the gap between sentence-level ConText markup and
document-level clinical meaning:

- A bare finding in "Past Medical History" → `HISTORICAL` without needing "history of".
- A bare finding in "Indication" → `INDICATION` without needing "rule out".
- A finding with an explicit modifier → the modifier wins over the section default.
- `ent._.cwyde_section_inherited` is `True` only when the section default was applied,
  making the provenance of every categorisation auditable.